<a href="https://colab.research.google.com/github/mjariganello/dmeyf2026/blob/main/notebooks_colab/420_ArbolesAzarosos.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4. Ensembles de Arboles de Decision

Un arbol de decisión es un modelo débil, el aumento del poder predictivo proviene al ensamblar varios arboles de decisión.
<br> Si promedio n arboles identicos, el resultados es exactamente el mismo que utilizar un solo arbol, necesito PERTURBAR cada arbol para disponer de variablidad

la variabilidad provendrá de estas fuentes:


*   Perturbar el dataset
*   Perturbar el algoritmo del arbol
*   Perturbar el dataset y el algoritmo del arbol al mismo tiempo

Se verán estos tres algoritmos


*   Arboles Azarosos
*   Random Forest
*   Gradient Boosting of Decision Trees

#### 4.01 Seteo del ambiente en Google Colab

Esta parte se debe correr con el runtime en Python3
<br>Ir al menu, Runtime -> Change Runtime Type -> Runtime type ->  **Python 3**

Conectar la virtual machine donde esta corriendo Google Colab con el  Google Drive, para poder tener persistencia de archivos

In [ ]:
# primero establecer el Runtime de Python 3
from google.colab import drive
drive.mount('/content/.drive')

Mounted at /content/.drive


Para correr la siguiente celda es fundamental en Arranque en Frio haber copiado el archivo kaggle.json al Google Drive, en la carpeta indicada en el instructivo

<br>los siguientes comando estan en shell script de Linux
*   Crear las carpetas en el Google Drive
*   "instalar" el archivo kaggle.json desde el Google Drive a la virtual machine para que pueda ser utilizado por la libreria  kaggle de Python
*   Bajar el  **dataset_pequeno**  al  Google Drive  y tambien al disco local de la virtual machine que esta corriendo Google Colab



In [ ]:
%%shell

mkdir -p "/content/.drive/My Drive/dmeyf"
mkdir -p "/content/buckets"
ln -sfn "/content/.drive/My Drive/dmeyf"   /content/buckets/b1

mkdir -p ~/.kaggle
cp /content/buckets/b1/kaggle/kaggle.json  ~/.kaggle
chmod 600 ~/.kaggle/kaggle.json


mkdir -p /content/buckets/b1/exp
mkdir -p /content/buckets/b1/datasets
mkdir -p /content/datasets


# defino funcion descargar()
descargar() {
  carpeta_destino="/content/buckets/b1/datasets/"
  url_origen="https://storage.googleapis.com/open-courses/utn2026-b40a/"
  archivo="$1"

  if ! test -f "$carpeta_destino""$archivo"; then
    wget  "$url_origen""$archivo"  -O "$carpeta_destino""$archivo"
  fi

  if ! test -f  "/content/datasets/""$archivo"; then
    cp  "$carpeta_destino""$archivo"  "/content/datasets/""$archivo"
  fi;
}

# hago la descarga efectiva, llamando a descargar()
descargar  "dataset_pequeno.csv"



---



## 4.02 Arboles Azarosos

Arboles Azarosos es el nombre de un algoritmo trivial (por favor NO confundir con Random Forest)
Qué tipo de perturbaciones se realizan en Arboles Azarosos
* Se perturba el dataset
* No se perturba el algoritmo, es siempre rpart original

Cada  arbolito de  Arboles Azarosos se entrena sobre un dataset perturbado,  que tiene exactamente la misma cantidad de registros pero solo un subconjunto de los atributos (campos)  del dataset, tomados al azar, de los originales.
<br> En esta primera corrida, se construira cada arbol en un dataset utilizando el 50% de los campos

Esta parte se debe correr con el runtime en lenguaje **R** Ir al menu, Runtime -> Change Runtime Type -> Runtime type -> R

limpio el ambiente de R

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Sun Aug 23 02:46:15 PM 2026"

In [ ]:
# limpio la memoria
rm(list=ls(all.names=TRUE)) # remove all objects
gc(full=TRUE, verbose=FALSE) # garbage collection

,used,(Mb),gc trigger,(Mb),max used,(Mb)
Ncells,671296,35.9,1473300,78.7,1473300,78.7
Vcells,1242676,9.5,8388608,64.0,1978712,15.1


In [ ]:
# cargo las librerias que necesito
require("data.table")
require("rpart")

Loading required package: data.table


Attaching package: ‘data.table’


The following object is masked from ‘package:base’:

    %notin%


Loading required package: rpart



Aqui debe cargar SU semilla primigenia

In [ ]:
PARAM <- list()
PARAM$semilla_primigenia <- 146023
PARAM$num_trees_max <- 32

# Vectores para la búsqueda en grilla (Grid Search)
v_feature_fraction <- c(0.2, 0.35, 0.5)
v_max_depth        <- c(8, 10, 12)
v_min_split        <- c(20, 60, 100)
v_min_bucket       <- c(3, 5, 8)

In [ ]:
# carpeta de trabajo
setwd("/content/buckets/b1/exp")
experimento <- "exp4023"
dir.create(experimento, showWarnings=FALSE)
setwd( paste0("/content/buckets/b1/exp/", experimento ))

In [ ]:
# lectura del dataset
dataset <- fread("/content/datasets/dataset_pequeno.csv")

In [ ]:
# defino los dataset de entrenamiento y aplicacion
dtrain <- dataset[foto_mes == 202107]
dfuture <- dataset[foto_mes == 202109]

# arreglo clase_ternaria por algun distraido ""
dfuture[, clase_ternaria := NA ]

In [ ]:
# Establezco cuales son los campos que puedo usar para la prediccion
# el copy() es por la Lazy Evaluation
campos_buenos <- copy(setdiff(colnames(dtrain), c("clase_ternaria")))

In [ ]:
# que tamanos de ensemble grabo a disco
grabar <- c(1, 2, 4, 8, 16, 32)

In [ ]:
tb_prediccion <- dfuture[, list(numero_de_cliente)]
# aqui se va acumulando la probabilidad del ensemble
tb_prediccion[, prob_acumulada := 0]

In [ ]:
set.seed(PARAM$semilla_primigenia) # Establezco la semilla aleatoria

In [ ]:
# --- Checkpoint para poder resumir si Colab se desconecta ---
# El archivo de control se guarda en el bucket (Google Drive), NO en el disco
# efimero de la VM, para que sobreviva a una reconexion.
archivo_avance <- "/content/buckets/b1/exp/exp4022/avance_combinaciones.txt"

if (file.exists(archivo_avance)) {
  combinaciones_hechas <- readLines(archivo_avance)
} else {
  combinaciones_hechas <- character(0)
}

cat("Combinaciones ya realizadas en corridas anteriores:", length(combinaciones_hechas), "\n")

Combinaciones ya realizadas en corridas anteriores: 56 


In [ ]:
for (v_ff in v_feature_fraction) {
  for (v_depth in v_max_depth) {
    for (v_split in v_min_split) {
      for (v_bucket in v_min_bucket) {

        # Filtro de coherencia
        if (v_bucket > (v_split / 2)) next

        # Identificador unico de esta combinacion (para el checkpoint)
        combo_id <- paste0("ff", v_ff, "_md", v_depth, "_ms", v_split, "_mb", v_bucket)

        # Si esta combinacion ya fue procesada en una corrida anterior, la salteo
        if (combo_id %in% combinaciones_hechas) {
          cat("Ya realizada, se saltea:", combo_id, "\n")
          next
        }

        # Envuelvo TODO el procesamiento de la combinacion en tryCatch
        # para que un error (de red, de kaggle, etc.) no corte el loop entero
        resultado <- tryCatch({

          # Asignación de parámetros a PARAM
          PARAM$feature_fraction <- v_ff
          PARAM$rpart$cp         <- -1
          PARAM$rpart$maxdepth   <- v_depth
          PARAM$rpart$minsplit   <- v_split
          PARAM$rpart$minbucket  <- v_bucket

          # Reiniciar predicción y fijar la semilla inicial
          tb_prediccion <- dfuture[, list(numero_de_cliente)]
          tb_prediccion[, prob_acumulada := 0]
          set.seed(PARAM$semilla_primigenia)

          # Generación de los 32 árboles para ESTA combinación
          for (arbolito in seq(PARAM$num_trees_max)) {
            qty_campos_a_utilizar <- as.integer(length(campos_buenos) * PARAM$feature_fraction)
            campos_random <- sample(campos_buenos, qty_campos_a_utilizar)
            campos_random <- paste(campos_random, collapse = " + ")
            formulita <- paste0("clase_ternaria ~ ", campos_random)

            modelo <- rpart(
              formulita,
              data = dtrain,
              xval = 0,
              control = PARAM$rpart
            )

            prediccion <- predict(modelo, dfuture, type = "prob")
            tb_prediccion[, prob_acumulada := prob_acumulada + prediccion[, "BAJA+2"]]
          }

          # Generación del archivo final tras completar los 32 árboles
          umbral_corte <- (1 / 40) * PARAM$num_trees_max
          tb_prediccion[, Predicted := as.numeric(prob_acumulada > umbral_corte)]

          archivo_kaggle <- paste0(
            "KA_ff", v_ff * 100,
            "_md", v_depth,
            "_ms", v_split,
            "_mb", v_bucket,
            ".csv"
          )

          fwrite(
            tb_prediccion[, list(numero_de_cliente, Predicted)],
            file = archivo_kaggle,
            sep = ","
          )

          # Subida a Kaggle
          comando <- "kaggle competitions submit"
          competencia <- "-c utn-2026-inicial"
          arch <- paste("-f", archivo_kaggle)
          mensaje <- paste0("-m 'ff=", v_ff, " cp=-1 md=", v_depth, " ms=", v_split, " mb=", v_bucket, "'")

          linea <- paste(comando, competencia, arch, mensaje)
          salida <- system(linea, intern = TRUE)
          cat("\nEnviado:", archivo_kaggle, "\nRespuesta Kaggle:", salida, "\n")

          # Si todo salio bien, marco esta combinacion como completada
          # (append=TRUE para no pisar lo ya escrito en corridas anteriores)
          write(combo_id, file = archivo_avance, append = TRUE)

          TRUE

        }, error = function(e) {
          # No marco combo_id como hecho: en la proxima corrida se reintenta
          cat("\nERROR en combinacion", combo_id, ":", conditionMessage(e), "\n")
          FALSE
        })

      }
    }
  }
}

In [ ]:
format(Sys.time(), "%a %b %d %X %Y")

[1] "Mon Aug 24 01:23:50 AM 2026"